In [1]:
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

# 사용자 동작(액션)을 구현하기 위한 클래스
from selenium.webdriver.common.action_chains import ActionChains

# 특정 엘리먼트에 마우스 휠 내리게 하는 동작
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
from tqdm import tqdm

import time
import re
def preprocess_sentence_kr(w):
    w = w.strip()
    w = re.sub(r"[^0-9가-힣?.!,]+" , " ",w)
    w = w.strip()
    return w

In [5]:
keyword = '음식물%20처리기%20사용%20후기%20'
term = 'from20240101to20250109'

# fstring으로 url에서 키워드와 검색일 설정 가능
blog_search_url = f"https://search.naver.com/search.naver?ssc=tab.blog.all&query={keyword}&sm=tab_opt&nso=so%3Ar%2Cp%3A{term}"

In [7]:
driver = wb.Chrome()
driver.get(blog_search_url)

In [15]:
# 실습1) 스크롤 30번 내리기 (1초 간격)

modal = driver.find_element(By.CSS_SELECTOR,'#container')

# scrollOrigin 생성 -> 그 스크롤 할 수 있는 영역 (모달영역)을 스크롤 영역으로 선택!
scroll_origin = ScrollOrigin.from_element(modal)
actions = ActionChains(driver) # actions 객체에 마우스와 키보드 동작을 체인으로 연결하여 실행

old_height = driver.execute_script("""
    const modal = document.querySelector('#container')
    return modal.scrollHeight

""")

count = 0 #스크롤 횟수
total_scrolls = 20


while count <= total_scrolls :
    # 스크롤 실행
    height = height + 20000
    actions.scroll_from_origin(scroll_origin,0,height).perform()
    # time
    time.sleep(1)
    

    # 새로운 높이 가져오기
    new_height = driver.execute_script("""
    const modal = document.querySelector('#container')
    return modal.scrollHeight
""")
    # 높이에 변화 없으면 종료 
    if (new_height == old_height):
        old_height = new_height
        break
    # 높이 업데이트
    else:
        count = count +1
    # 변화 있으면 count 횟수 증가


In [61]:
# 실습2) 블로그 url 주소 수집 후 href_list에 저장하기 

#title_link
href_list =[]
blog_list = driver.find_elements(By.CSS_SELECTOR,'.title_link')

In [67]:
for i in blog_list :   
    blog_href = i.get_attribute('href')
    if (blog_href[8:12] == "blog"):
        href_list.append(blog_href)

In [68]:
href_list

['https://blog.naver.com/next200208/223472945242',
 'https://blog.naver.com/lje1049/223679643524',
 'https://blog.naver.com/tinker_bell4/223674096057',
 'https://blog.naver.com/sosin279/223703912417',
 'https://blog.naver.com/besisi1004/223590373932',
 'https://blog.naver.com/maylily_baby/223716415449',
 'https://blog.naver.com/cheawhi/223660694759',
 'https://blog.naver.com/re14333/223677281816',
 'https://blog.naver.com/lalahonghong/223628605257',
 'https://blog.naver.com/cjs0308cjs/223714902608',
 'https://blog.naver.com/mingki0727/223714112612',
 'https://blog.naver.com/iclimb/223648184400',
 'https://blog.naver.com/kkwrkdrk2/223693660666',
 'https://blog.naver.com/kut_da_92/223558532889',
 'https://blog.naver.com/mybombom/223625512043',
 'https://blog.naver.com/fevernova22/223649736167',
 'https://blog.naver.com/coconeldeco/223698245564',
 'https://blog.naver.com/jinahnim/223388866868',
 'https://blog.naver.com/besisi1004/223694939381',
 'https://blog.naver.com/kimhjz1/22370299731

In [71]:
len(href_list)

654

In [77]:
# 저장한 url list로 접근
driver.get(href_list[0])

# iframe으로 전환하기 
driver.switch_to.frame('mainFrame')

In [83]:
# 블로그 내용 가져오기
content = driver.find_elements(By.CSS_SELECTOR,".se-main-container")

In [95]:
f = open('블로그 리뷰 데이터.txt', 'w')

for i in tqdm(range(len(href_list))):
    driver.get(href_list[i])
    print(href_list[i])

    time.sleep(2)

    driver.switch_to.frame('mainFrame')
    try:
        # 본문에 image가 있으면 지워라 
        driver.execute_script("""
            const imageBox = document.querySelectorAll("div.se-image");
            for (let i =0 ; i<imageBox.length; i++){
                imageBox[i].remove()
            }
        """)

    except:
        pass

    try:
        # 블로그의 형태가 여러개가 있을 수 있음
        # 경우의 수 1
        content = driver.find_element(By.CLASS_NAME, 'se-main-container')
    except:
        # 안되면 경우의 수 2
        content = driver.find_element(By.CSS_SELECTOR, 'se-component-wrap.sect_dsc')

    content = preprocess_sentence_kr(content.text)
    f.write(content)
f.close()
driver.close()

  0%|                                                                                          | 0/654 [00:00<?, ?it/s]

https://blog.naver.com/next200208/223472945242


  0%|▏                                                                                 | 1/654 [00:05<54:56,  5.05s/it]

https://blog.naver.com/lje1049/223679643524


  0%|▏                                                                               | 2/654 [00:12<1:10:00,  6.44s/it]

...

https://blog.naver.com/theo_philus/223505759629


  4%|██▉                                                                              | 24/654 [02:23<56:41,  5.40s/it]

https://blog.naver.com/zooty7979/223707832732


  4%|███                                                                            | 25/654 [02:32<1:03:57,  6.10s/it]


KeyboardInterrupt: 